In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
import joblib


In [ ]:
td = pd.read_csv('train_data.csv', sep=',')  # train data
tsd = pd.read_csv('test_data.csv', sep=',')  # test data
td.shape


In [ ]:
td.drop_duplicates(inplace=True)  # if there is
tsd.drop_duplicates(inplace=True)  # if there is
td.shape


In [ ]:
# Clean all object columns in one go
def clean_dataframe(df):

    # remove spaces from column names
    df.columns = df.columns.str.strip()

    # remove duplicated rows
    df.drop_duplicates(inplace=True)

    return df


td = clean_dataframe(td)
tsd = clean_dataframe(tsd)

td.head()


In [ ]:
sneaky_nulls = ['?', 'unknown', 'N/A', 'none', 'None', '-', '']

td.replace(sneaky_nulls, np.nan, inplace=True)
tsd.replace(sneaky_nulls, np.nan, inplace=True)

# print(td.isnull().sum())
# print(tsd.isnull().sum())
td.head()


In [ ]:
td.drop(columns='id', inplace=True)
tsd.drop(columns='id', inplace=True)
td.shape


## Change 1: Class imbalance check
Before anything else, check whether the target is balanced or not, since this affects metric choice and `class_weight`.

In [ ]:
# Check class distribution of the target
target_col = td.columns[-1]
print("Class distribution (Train):")
print(td[target_col].value_counts(normalize=True))
print()
print("Class distribution (Test):")
print(tsd[target_col].value_counts(normalize=True))


In [ ]:
# features and target
X = td.iloc[:, :-1]
y = td.iloc[:, -1]

# test dataset
X_test = tsd.iloc[:, :-1]
y_test = tsd.iloc[:, -1]

# split training data into train and validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

X_train.shape, X_val.shape


**Change 2:** Added `stratify=y` to `train_test_split` to keep class ratios consistent between train and validation (important when the data is imbalanced).

In [ ]:
num_cols = X_train.select_dtypes(include="number").columns
cat_cols = X_train.select_dtypes(exclude="number").columns

print(cat_cols)

imputer = ColumnTransformer(transformers=[
    ('cat', SimpleImputer(strategy='most_frequent'), cat_cols),
    ('num', SimpleImputer(strategy='mean'), num_cols)
], remainder='passthrough')

imputer.set_output(transform="pandas")

X_train = imputer.fit_transform(X_train)
X_val = imputer.transform(X_val)
X_test = imputer.transform(X_test)


X_train.head()


In [ ]:
# categorical columns in your dataset after imputation
nominal_cols = ['cat__Gender', 'cat__work_type', 'cat__smoking_status']

# One Hot Encoding for categorical features
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe.fit(X_train[nominal_cols])

for split_X in [X_train, X_val, X_test]:
    encoded = ohe.transform(split_X[nominal_cols])

    encoded_df = pd.DataFrame(
        encoded,
        columns=ohe.get_feature_names_out(nominal_cols),
        index=split_X.index
    )

    split_X.drop(columns=nominal_cols, inplace=True)

    for col in encoded_df.columns:
        split_X[col] = encoded_df[col]

# Encode target column
le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)
y_test = le.transform(y_test)

y_train


## Change 3: GridSearchCV instead of a single-parameter loop
Train and validation are merged and 5-fold Cross-Validation picks the best hyperparameters, instead of relying on one validation split. Multiple parameters are tuned together (not just `max_depth`), and `class_weight='balanced'` is added to handle any class imbalance.

In [ ]:
# Merge train + val so Cross-Validation can use all available training data
X_train_full = pd.concat([X_train, X_val])
y_train_full = np.concatenate([y_train, y_val])

param_grid = {
    'max_depth': [3, 5, 7, 9, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy'],
    'class_weight': [None, 'balanced']
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    scoring='f1_weighted',
    cv=5,
    n_jobs=-1
)

print("--- Hyperparameter Tuning (5-Fold Cross-Validation) ---")
grid_search.fit(X_train_full, y_train_full)

print(f"Best Params: {grid_search.best_params_}")
print(f"Best CV F1 Score (weighted): {grid_search.best_score_:.4f}")

final_model = grid_search.best_estimator_


## Change 4: Final model evaluation on the test set + ROC-AUC
ROC-AUC is added alongside accuracy and F1, since it gives a more reliable picture on imbalanced problems where accuracy can be misleading.

In [ ]:
print("--- Final Model Evaluation (Test Set) ---")

# Predict on the strictly held-out test set
y_test_pred = final_model.predict(X_test)
y_test_proba = final_model.predict_proba(X_test)[:, 1]

# Calculate final metrics
test_accuracy = accuracy_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred, average='weighted')
test_auc = roc_auc_score(y_test, y_test_proba)
conf_matrix = confusion_matrix(y_test, y_test_pred)

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test F1 (weighted): {test_f1:.4f}")
print(f"Test ROC-AUC: {test_auc:.4f}\n")
print("Classification Report (Precision, Recall, F1):")
print(classification_report(y_test, y_test_pred))


In [ ]:
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Test Set Confusion Matrix\n(Best Model)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')

# Feature importance chart, replacing the old max_depth-vs-F1 plot
plt.subplot(1, 2, 2)
importances = pd.Series(final_model.feature_importances_, index=X_train.columns)
importances.sort_values(ascending=False).head(10).plot(kind='barh', color='#1f77b4')
plt.title('Top 10 Feature Importances')
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()


## Change 5: Compare Decision Tree against other models
A single Decision Tree is prone to overfitting. It is compared here with Random Forest and Logistic Regression on the same test set.

In [ ]:
models = {
    'Decision Tree (Tuned)': final_model,
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced', n_jobs=-1),
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced')
}

results = []
for name, model in models.items():
    if name != 'Decision Tree (Tuned)':
        model.fit(X_train_full, y_train_full)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'F1 (weighted)': f1_score(y_test, pred, average='weighted'),
        'ROC-AUC': roc_auc_score(y_test, proba)
    })

results_df = pd.DataFrame(results).sort_values('F1 (weighted)', ascending=False)
results_df


## Change 6: Save the model and encoders
So the model can be reused later without retraining.

In [ ]:
joblib.dump(final_model, 'final_model.pkl')
joblib.dump(ohe, 'encoder.pkl')
joblib.dump(le, 'label_encoder.pkl')
joblib.dump(imputer, 'imputer.pkl')

print("Model and encoders saved successfully.")
